# TP2 - Partie 7 : Embeddings statiques + tête dense

## Importations

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

## Charger le modèle BERT pour extraire les embeddings

In [ ]:
model_name = "aubmindlab/bert-base-arabertv02-twitter"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

## Fonction d'extraction des embeddings

In [ ]:
def extract_embeddings(texts: list):
    """Extraire les embeddings (premier token) pour chaque texte."""
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()

    return embeddings

## Entraîner une tête dense (Logistic Regression)

In [ ]:
X_train_embeds = extract_embeddings(df_train['cleaned_tweet'].tolist())
X_test_embeds = extract_embeddings(df_test['cleaned_tweet'].tolist())

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_embeds)
X_test_scaled = scaler.transform(X_test_embeds)

model_dense = LogisticRegression(random_state=42)
model_dense.fit(X_train_scaled, df_train['sarcasm'])

## Prédiction

In [ ]:
y_pred = model_dense.predict(X_test_scaled)

## Sauvegarde des prédictions

In [ ]:
df_result = pd.DataFrame({
    'id': df_test['id'],
    'y_pred': y_pred
})
df_result.to_csv("predictions/static_seed42.csv", index=False)

print("✅ Prédictions sauvegardées dans `predictions/static_seed42.csv`.")